# Initial comparison: bagging vs random forest

This notebook compares a standard bagging ensemble with a random forest using the same sparse one-hot features. It uses a reproducible sample for an initial feasibility comparison; the untouched test partition is not used.

In [2]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from model_data import prepare_model_data
from model_evaluation import macro_f1_test

## Prepare a common sample

The shared preprocessing is fitted on the full training partition only. Both models then use the same deterministic sample from the chronological training and validation partitions.

In [3]:
model_data = prepare_model_data(PROJECT_ROOT / "data")

train_size = min(500_000, model_data.X_train.shape[0])
validation_size = min(250_000, model_data.X_validation.shape[0])

X_train = model_data.X_train[:train_size]
y_train = model_data.y_train.iloc[:train_size]
X_validation = model_data.X_validation[:validation_size]
y_validation = model_data.y_validation.iloc[:validation_size]

pd.DataFrame({
    "partition": ["train", "validation"],
    "rows": [len(y_train), len(y_validation)],
    "features": [X_train.shape[1], X_validation.shape[1]],
})

,partition,rows,features
0,train,500000,4758
1,validation,250000,4758


## Train and compare the two ensembles

Both candidates use 20 trees and the same maximum depth. The only intended difference is the ensemble method: bootstrap aggregation of decision trees versus random forests with feature subsampling.

In [5]:
models = {
    "bagging": BaggingClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=50,
            class_weight="balanced",
            random_state=42,
        ),
        n_estimators=100,
        max_samples=1.0,
        bootstrap=True,
        n_jobs=-1,
        random_state=42,
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=50,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42,
    ),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    results.append(
        macro_f1_test(
            model,
            X_validation,
            y_validation,
            model_name=name,
        )
    )

comparison = (
    pd.DataFrame(results)
    .sort_values(["macro_f1", "accuracy"], ascending=False)
    .reset_index(drop=True)
)
comparison

,model,n_test,macro_f1,accuracy
0,random_forest,250000,0.293953,0.357008
1,bagging,250000,0.282480,0.354640


Random Forest currently performs better on the validation sample, with macro-F1 of 0.2940 versus 0.2825 for bagging. However, the training cost of both tree ensembles is not justified by their performance compared with the project’s other models. This is expected for sparse, high-dimensional one-hot features, so neither bagging nor Random Forest will be used in the final model comparison.